# Mission 3: Alpha Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_03_alpha_discovery/notebook.ipynb)

**Learning objectives**
- Frame signal search as a multiple-testing problem and apply corrections
- Use walk-forward IC to validate a signal before committing to it
- Understand signal decay and its implications for turnover and TC
- Build a multi-signal composite that is robust OOS

---

## Background

The synthetic market has several features. Most are pure noise — random variables with no predictive power. One or two are **planted alphas**: they have a small but genuine predictive relationship with next-day returns.

Your job is to find them *without overfitting*. This is harder than it sounds:

- If you test 10 features and pick the best one, you expect the best to look good *even if all 10 are noise* (multiple testing bias).
- A signal that works well in-sample may be capturing a regime that no longer holds OOS.
- Transaction costs can erase alpha that looks significant in a frictionless backtest.

We'll work through a rigorous discovery pipeline that addresses all three.

---

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi-lab>=0.1.0 statsmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

from convexpi.lab import SyntheticMarket, Backtest, Grader

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready.')

In [ ]:
market = SyntheticMarket(n_assets=200, n_days=1000, seed=42)
FEATURE_NAMES = market.FEATURE_NAMES
N_FEATURES    = len(FEATURE_NAMES)
TRAIN_END     = market.train_end

train_features = market.features[:TRAIN_END]   # (T_train, N, F)
train_returns  = market.returns[:TRAIN_END]    # (T_train, N)

print(f"Features : {FEATURE_NAMES}")
print(f"Train    : {TRAIN_END} days  |  Holdout: {market.n_days - TRAIN_END} days")

---
## Part 1: Naive Search — and Why It Fails

First, let's do it the wrong way: compute IS IC for every feature, rank them, and pick the winner.

In [ ]:
def compute_daily_ics(features_3d, returns_2d, lag=1):
    """Returns DataFrame of daily IC per feature. Shape: (T-lag, n_features)."""
    T = len(features_3d) - lag
    records = []
    for t in range(T):
        row = {}
        for fi, fname in enumerate(FEATURE_NAMES):
            signal  = features_3d[t, :, fi]
            fwd_ret = returns_2d[t + lag]
            ic, _   = stats.spearmanr(signal, fwd_ret)
            row[fname] = ic
        records.append(row)
    return pd.DataFrame(records)

print('Computing IS ICs...')
ic_df = compute_daily_ics(train_features, train_returns)

# Summary stats
summary = pd.DataFrame({
    'mean_IC':  ic_df.mean(),
    'std_IC':   ic_df.std(),
    'IC-IR':    ic_df.mean() / ic_df.std() * np.sqrt(252),
    't_stat':   stats.ttest_1samp(ic_df, 0).statistic,
    'p_value':  stats.ttest_1samp(ic_df, 0).pvalue,
})
print("\nNaive IS IC summary:")
print(summary.sort_values('mean_IC', ascending=False).to_string(float_format='{:.4f}'.format))

**Exercise 1.1** — Which feature has the highest IS IC? Is its p-value significant at p < 0.05?

Now apply a **multiple testing correction** (Benjamini-Hochberg FDR). With N features tested, some will look significant by chance.

In [ ]:
reject, p_adj, _, _ = multipletests(summary['p_value'], alpha=0.05, method='fdr_bh')
summary['p_adj_BH']  = p_adj
summary['significant'] = reject

print("After Benjamini-Hochberg FDR correction (q=0.05):")
print(summary[['mean_IC', 'IC-IR', 'p_value', 'p_adj_BH', 'significant']]
      .sort_values('mean_IC', ascending=False)
      .to_string(float_format='{:.4f}'.format))

**Exercise 1.2** — How many features survive FDR correction? Does the naive winner still look significant after the correction?

---
## Part 2: Walk-Forward IC Validation

A static IS IC test conflates regime effects with genuine alpha. Walk-forward validation asks: is the IC *consistently* positive across many sub-periods, or is it driven by a single lucky stretch?

In [ ]:
def walk_forward_ic(features_3d, returns_2d, feature_idx: int,
                    train_window: int = 120, step: int = 20):
    """Rolling window IC — fit on train_window days, evaluate on next step days."""
    T = len(features_3d) - 1
    results = []
    pos = train_window
    while pos + step <= T:
        oos_ics = []
        for t in range(pos, pos + step):
            signal  = features_3d[t, :, feature_idx]
            fwd_ret = returns_2d[t + 1]
            ic, _   = stats.spearmanr(signal, fwd_ret)
            oos_ics.append(ic)
        results.append({'start': pos, 'end': pos + step, 'mean_ic': np.mean(oos_ics)})
        pos += step
    return pd.DataFrame(results)


fig, axes = plt.subplots(N_FEATURES, 1, figsize=(11, 2.2 * N_FEATURES), sharex=True)

for i, fname in enumerate(FEATURE_NAMES):
    wf = walk_forward_ic(train_features, train_returns, i)
    ax = axes[i] if N_FEATURES > 1 else axes
    ax.bar(wf['start'], wf['mean_ic'], width=wf['end'] - wf['start'],
           color=['steelblue' if v > 0 else 'tomato' for v in wf['mean_ic']], alpha=0.8)
    ax.axhline(0, color='black', lw=0.5)
    n_pos = (wf['mean_ic'] > 0).sum()
    ax.set_title(f'{fname}  — positive in {n_pos}/{len(wf)} windows')

plt.suptitle('Walk-forward IC (20-day OOS windows)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

**Exercise 2.1** — Which feature is positive in the most walk-forward windows? Does this match the IS IC ranking?

**Exercise 2.2** — Is there a feature that looks good in the first half of the training period but flat in the second? What does this suggest about regime stationarity?

---
## Part 3: Signal Decay and Turnover

Alpha signals decay: a rank today may not predict returns two days from now. Measuring decay tells you how often to trade — and therefore your transaction cost burden.

In [ ]:
def signal_decay(features_3d, returns_2d, feature_idx: int, max_lag: int = 10):
    """IC at each lag from 1 to max_lag."""
    ics = []
    for lag in range(1, max_lag + 1):
        ic_vals = []
        for t in range(len(features_3d) - lag):
            ic, _ = stats.spearmanr(features_3d[t, :, feature_idx], returns_2d[t + lag])
            ic_vals.append(ic)
        ics.append({'lag': lag, 'mean_ic': np.mean(ic_vals)})
    return pd.DataFrame(ics)


fig, ax = plt.subplots(figsize=(10, 4))
for i, fname in enumerate(FEATURE_NAMES):
    decay = signal_decay(train_features, train_returns, i)
    ax.plot(decay['lag'], decay['mean_ic'], marker='o', ms=4, label=fname)

ax.axhline(0, color='grey', lw=0.5)
ax.set_title('Signal decay: IC vs. forward lag')
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Mean IC')
ax.legend()
plt.tight_layout()
plt.show()

**Exercise 3.1** — At what lag does each signal's IC become indistinguishable from zero? This is the signal's **half-life** — it determines how often you need to rebalance.

**Exercise 3.2** — If transaction costs are 10 bps per trade and the signal has a 1-day half-life, roughly how much annual alpha (in bps) do you need to break even? *Hint: daily IC × sqrt(252) ≈ annualized IC-IR; multiply by signal vol.*

---
## Part 4: Building a Robust Composite Signal

If multiple signals survive the tests above, combining them can improve the IC-IR (diversification of signal noise). The key is to weight signals by their walk-forward IC-IR — not their IS IC.

In [ ]:
# Compute walk-forward IC-IR for each feature to use as weights
wf_icirs = {}
for i, fname in enumerate(FEATURE_NAMES):
    wf = walk_forward_ic(train_features, train_returns, i)
    icir = wf['mean_ic'].mean() / (wf['mean_ic'].std() + 1e-9)
    wf_icirs[fname] = max(0.0, icir)  # only include positive contributors

total_icir = sum(wf_icirs.values())
weights = {k: v / (total_icir + 1e-9) for k, v in wf_icirs.items()}

print("Walk-forward IC-IR weights:")
for fname, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {fname}: {w:.3f}  (raw IC-IR: {wf_icirs[fname]:.3f})")

In [ ]:
class CompositeStrategy:
    """Weighted combination of features, weights from walk-forward IC-IR."""
    def __init__(self, weights: dict):
        self.weights = weights

    def predict(self, features: np.ndarray) -> np.ndarray:
        signal = np.zeros(features.shape[0])
        for fi, fname in enumerate(FEATURE_NAMES):
            w = self.weights.get(fname, 0.0)
            if w > 0:
                raw = features[:, fi]
                # Cross-sectional z-score before combining
                z = (raw - raw.mean()) / (raw.std() + 1e-8)
                signal += w * z
        return signal


# Backtest the composite IS
composite = CompositeStrategy(weights)
bt = Backtest(market, composite, top_k=20, transaction_cost_bps=10)
result = bt.run()

print(f"Composite IS Sharpe      : {result.sharpe:.3f}")
print(f"Composite IS Annual Ret  : {result.annualized_return:.2%}")

---
## Part 5: Submit to the Grader

Build your final submission. The grader will evaluate it on the hidden holdout period.

In [ ]:
# Render weights as a Python literal for the submission
weights_items = ', '.join(f'{k!r}: {v:.6f}' for k, v in weights.items())

submission_code = f'''\
import numpy as np

FEATURE_NAMES = {FEATURE_NAMES!r}
_WEIGHTS = {{{weights_items}}}

class MyStrategy:
    """
    Walk-forward IC-IR weighted composite of synthetic market features.
    Each feature is cross-sectionally z-scored before combination.
    Weights are proportional to OOS IC-IR estimated on rolling windows.
    """
    def predict(self, features: np.ndarray) -> np.ndarray:
        signal = np.zeros(features.shape[0])
        for fi, fname in enumerate(FEATURE_NAMES):
            w = _WEIGHTS.get(fname, 0.0)
            if w > 0:
                raw = features[:, fi]
                z = (raw - raw.mean()) / (raw.std() + 1e-8)
                signal += w * z
        return signal
'''

print("=== Paste into the ConvexityLabs web editor ===")
print(submission_code)

---
## Part 6: Challenges

**Challenge A (Easy):** The composite above weights features by IC-IR. Try weighting by *mean IC* instead. Which gives a better OOS Sharpe and why?

**Challenge B (Medium):** Some features may have a **sign flip** OOS — a feature with positive IS IC may have negative OOS IC (regime change). Implement a test: if a feature's IC is negative in the last 60 training days, set its weight to zero. Does this improve robustness?

**Challenge C (Medium):** Estimate the **optimal holding period** for each signal by finding the lag at which IC × (1 - 2×TC/IC) is maximised. Implement a strategy that rebalances at this frequency instead of daily.

**Challenge D (Hard):** Implement a **shrinkage estimator** for the IC vector: instead of using the raw walk-forward mean IC as the weight, shrink it toward zero by a factor proportional to the estimation uncertainty. Does this help OOS?

In [ ]:
# Challenge B starter — recency filter
RECENCY_WINDOW = 60

recency_ics = {}
for i, fname in enumerate(FEATURE_NAMES):
    recent_ics = []
    for t in range(TRAIN_END - RECENCY_WINDOW - 1, TRAIN_END - 1):
        ic, _ = stats.spearmanr(train_features[t, :, i], train_returns[t + 1])
        recent_ics.append(ic)
    recency_ics[fname] = np.mean(recent_ics)

print("Recent IC (last 60 training days):")
for fname, ic in sorted(recency_ics.items(), key=lambda x: -x[1]):
    flag = "✓ keep" if ic > 0 else "✗ drop"
    print(f"  {fname}: {ic:.4f}  {flag}")

filtered_weights = {k: v if recency_ics.get(k, 0) > 0 else 0.0 for k, v in weights.items()}
total = sum(filtered_weights.values())
if total > 0:
    filtered_weights = {k: v / total for k, v in filtered_weights.items()}
print("\nFiltered + renormalised weights:", {k: round(v, 3) for k, v in filtered_weights.items()})

---
## Wrap-up

Key takeaways:

1. **Multiple testing is unavoidable.** Any time you test more than one hypothesis on the same data, you need a correction. Benjamini-Hochberg controls the false discovery rate at a known level; naive p < 0.05 does not.

2. **Walk-forward consistency beats IS magnitude.** A signal with moderate IS IC that is positive in 80% of rolling windows is far more trustworthy than one with high IS IC concentrated in a single lucky stretch.

3. **Signal decay determines your cost structure.** A short-lived signal requires frequent rebalancing. If TC > alpha, you cannot trade profitably even if the signal is real.

4. **Combine with humility.** Weight signals by their *estimated* predictive power, not their in-sample realisation. Shrinkage and recency filters both push the weights toward the prior that "most signals are noise".

---

*You've completed all three ConvexityLabs missions. Your journey from overfitting → market-making → alpha discovery is the core of quantitative research. The professionals do the same thing — with more data, more compute, and higher stakes.*